In [ ]:
# This mounts your Google Drive to the Colab VM.
from google.colab import drive
drive.mount('/content/drive')

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
#import sys
#sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))


Mounted at /content/drive


In [ ]:
%cd /content/drive/My Drive/Python/JHU_553.640/DLNS/

/content/drive/My Drive/Python/JHU_553.640/DLNS


In [ ]:
# Import libraries and modules
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import matplotlib.dates as mdates
from sklearn.metrics import mean_squared_error
import random
import os

# Import your custom modules
from YCdataset import YieldCurveDataset
from encoder import CNN1DTransformerEncoder, TransformerOnlyEncoder, CNN1DRNNEncoder, CNN1DLSTMEncoder
from NS_layer import NelsonSiegelLayer
from DLNS import DLNS_CNNTransformer, DLNSTransformerOnly, DLNS_CNNRNN, DLNS_CNNLSTM
from train import train_model, plot_training_history, average_model_parameters
from evaluation import evaluate_model, denormalize_and_evaluate, multi_step_forecast, evaluate_multi_step_forecast, extract_factors_and_lambdas
from visualization import plot_yield_curves,plot_factors_over_time, plot_yield_curve_3d, heatmap_predictions_error, plot_factors_over_time_with_periods

# multiple output in notebook without print()
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Other options
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('mode.chained_assignment', None)

'''
# Set the seed
# Define a global seed value # 30
SEED = 42

# Set random seeds for Python, NumPy, PyTorch
def set_all_seeds(seed=SEED):
    # Python's built-in random
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For multi-GPU setups

    # Additional PyTorch settings for deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Set environment variable for other libraries
    os.environ['PYTHONHASHSEED'] = str(seed)

    print(f"All random seeds set to {seed}")

# Call this function at the beginning of your code
set_all_seeds()
'''


'\n# Set the seed\n# Define a global seed value # 30\nSEED = 42\n\n# Set random seeds for Python, NumPy, PyTorch\ndef set_all_seeds(seed=SEED):\n    # Python\'s built-in random\n    random.seed(seed)\n\n    # NumPy\n    np.random.seed(seed)\n\n    # PyTorch\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)  # For multi-GPU setups\n\n    # Additional PyTorch settings for deterministic behavior\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n    # Set environment variable for other libraries\n    os.environ[\'PYTHONHASHSEED\'] = str(seed)\n\n    print(f"All random seeds set to {seed}")\n\n# Call this function at the beginning of your code\nset_all_seeds()\n'

### Split train/val/test dataset and Normalization

In [ ]:
# Data loading
# Load the processed monthly data from the csv file, using 'Date' as the index
df = pd.read_csv('df_monthly.csv', index_col='Date')

# Rename the '3M' and '6M' columns
df = df.rename(columns={'3M': '0.25Y', '6M': '0.5Y'})

# Define train/val size
# Calculate the size of the training and validation datasets
#train_size = int(0.765 * len(df)) # 76.5% of the data for training
#val_size = int(0.153 * len(df)) # 15.3% of the data for validation
train_size = int(0.80 * len(df)) # 76.5% of the data for training
val_size = int(0.07 * len(df)) # 15.3% of the data for validation

# parmeters
lookback_window = 6
pred_h = 6
macro_vars = 4

## Create datasets with macro variables
# train dataset
train_dataset_with_macro = YieldCurveDataset(
    df.iloc[:train_size],
    seq_length=lookback_window,
    pred_horizon=pred_h,
    is_train=True,
    use_macro=True
)
yield_scaler = train_dataset_with_macro.yield_scaler # Extract the fitted yield scaler
macro_scaler = train_dataset_with_macro.macro_scaler # Extract the fitted macro scaler

# validation dataset
val_dataset_with_macro = YieldCurveDataset(
    df.iloc[train_size:train_size+val_size],
    seq_length=lookback_window,
    pred_horizon=pred_h,
    yield_scaler=yield_scaler,
    macro_scaler=macro_scaler,
    is_train=False,
    use_macro=True
)

# test dataset
test_dataset_with_macro = YieldCurveDataset(
    df.iloc[train_size+val_size:],
    seq_length=lookback_window,
    pred_horizon=pred_h,
    yield_scaler=yield_scaler,
    macro_scaler=macro_scaler,
    is_train=False,
    use_macro=True
)

## Create datasets without macro variables
# train dataset
train_dataset_without_macro = YieldCurveDataset(
    df.iloc[:train_size],
    seq_length=lookback_window,
    pred_horizon=pred_h,
    is_train=True,
    use_macro=False
)
yield_scaler_no_macro = train_dataset_without_macro.yield_scaler

# validation dataset
val_dataset_without_macro = YieldCurveDataset(
    df.iloc[train_size:train_size+val_size],
    seq_length=lookback_window,
    pred_horizon=pred_h,
    yield_scaler=yield_scaler_no_macro,
    is_train=False,
    use_macro=False
)

# test dataset
test_dataset_without_macro = YieldCurveDataset(
    df.iloc[train_size+val_size:],
    seq_length=lookback_window,
    pred_horizon=pred_h,
    yield_scaler=yield_scaler_no_macro,
    is_train=False,
    use_macro=False
)


### DLNS Implementation

In [ ]:
# Function to fix seed
SEED=30
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Create data loaders
batch_size = 16 # check point: why 32?, Should I use different batch_size for validation and test?
train_loader = DataLoader(train_dataset_with_macro, batch_size=batch_size, shuffle=True, worker_init_fn=seed_worker,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset_with_macro, batch_size=batch_size, worker_init_fn=seed_worker,
                          generator=torch.Generator().manual_seed(SEED))
test_loader = DataLoader(test_dataset_with_macro, batch_size=batch_size, worker_init_fn=seed_worker,
                          generator=torch.Generator().manual_seed(SEED))

# Define model parameters
input_dim = train_dataset_with_macro[0][0].shape[1]  # Total number of features (yields + macro)
yield_dim = len(train_dataset_with_macro.yield_cols)  # Number of yield maturities
macro_dim = len(train_dataset_with_macro.macro_cols)  # Number of macro features
seq_length = train_dataset_with_macro.seq_length
maturities = [float(col.replace('Y', '')) for col in train_dataset_with_macro.yield_cols]  # Extract maturities from column names

# Create models (time-invariant lambda)
cnn_transformer_model = DLNS_CNNTransformer(
    input_dim=input_dim,
    yield_dim=yield_dim,
    macro_dim=macro_dim,
    maturities=maturities,
    hidden_dim=64,
    time_varying_decay=True,
    cnn_out_channels=32,
    cnn_kernel_size=3,
    transformer_dim=64,
    nhead=4,
    num_transformer_layers=2,
    dropout=0.1,
    seq_length=seq_length,
    use_macro=True
)

transformer_only_model = DLNSTransformerOnly(
    input_dim=input_dim,
    maturities=maturities,
    hidden_dim=64,
    time_varying_decay=True,
    transformer_dim=64,
    nhead=4,
    num_transformer_layers=2,
    dropout=0.1,
    seq_length=seq_length

)

cnn_rnn_model = DLNS_CNNRNN(
    input_dim=input_dim,
    yield_dim=yield_dim,
    macro_dim=macro_dim,
    maturities=maturities,
    hidden_dim=64,
    time_varying_decay=True,
    cnn_out_channels=32,
    cnn_kernel_size=3,
    rnn_hidden_dim=64,
    num_rnn_layers=2,
    dropout=0.1,
    seq_length=seq_length,
    use_macro=True,
    bidirectional=False
)


cnn_lstm_model = DLNS_CNNLSTM(
    input_dim=input_dim,
    yield_dim=yield_dim,
    macro_dim=macro_dim,
    maturities=maturities,
    hidden_dim=64,
    time_varying_decay=True,
    cnn_out_channels=32,
    cnn_kernel_size=3,
    lstm_hidden_dim=64,
    num_lstm_layers=2,
    dropout=0.1,
    seq_length=seq_length,
    use_macro=True
)


In [ ]:
## Train models
# CNN-Transformer model
cnn_transformer_model, cnn_history = train_model(
    cnn_transformer_model,
    train_loader,
    val_loader,
    criterion=torch.nn.MSELoss(),
    learning_rate=0.001,
    weight_decay=1e-5,
    n_epochs=100,
    patience=30,
    model_path='cnn_transformer_model.pth'
)


In [ ]:
# Plot training history
plot_training_history(cnn_history)

In [ ]:
# Transformer-only model
transformer_only_model, transformer_history = train_model(
    transformer_only_model,
    train_loader,
    val_loader,
    criterion=torch.nn.MSELoss(),
    learning_rate=0.001,
    weight_decay=1e-5,
    n_epochs=100,
    patience=30,
    model_path='transformer_only_model.pth'
)

In [ ]:
# Plot training history
plot_training_history(transformer_history)

In [ ]:
## Train models
# CNN-RNN Model
cnn_rnn_model, rnn_history = train_model(
    cnn_rnn_model,
    train_loader,
    val_loader,
    criterion=torch.nn.MSELoss(),
    learning_rate=0.001,
    weight_decay=1e-5,
    n_epochs=100,
    patience=30,
    model_path='cnn_rnn_model.pth'
)


In [ ]:
# Plot training history
plot_training_history(rnn_history)

In [ ]:
## Train models
# CNN-LSTM Model
cnn_lstm_model, lstm_history = train_model(
    cnn_lstm_model,
    train_loader,
    val_loader,
    criterion=torch.nn.MSELoss(),
    learning_rate=0.001,
    weight_decay=1e-5,
    n_epochs=100,
    patience=30,
    model_path='cnn_lstm_model.pth'
)


In [ ]:
# Plot training history
plot_training_history(lstm_history)

In [ ]:
# Step 5: Evaluate models
# CNN-Transformer model
cnn_results = denormalize_and_evaluate(
    cnn_transformer_model,
    test_loader,
    yield_scaler,
    maturities
)

# Transformer-only model
transformer_results = denormalize_and_evaluate(
    transformer_only_model,
    test_loader,
    yield_scaler,
    maturities
)


# CNN-RNN model
rnn_results = denormalize_and_evaluate(
    cnn_rnn_model,
    test_loader,
    yield_scaler,
    maturities
)

# CNN-LSTM model
lstm_results = denormalize_and_evaluate(
    cnn_lstm_model,
    test_loader,
    yield_scaler,
    maturities
)


In [ ]:
# Step 6: Compare model performance
print(f"CNN-Transformer Model RMSE: {cnn_results['rmse']:.6f}")
print(f"Transformer-Only Model RMSE: {transformer_results['rmse']:.6f}")
print(f"CNN-RNN Model RMSE: {rnn_results['rmse']:.6f}")
print(f"CNN-LSTM Model RMSE: {lstm_results['rmse']:.6f}")

print(f"{cnn_results['rmse']:.2f}", '&', f"{transformer_results['rmse']:.2f}", '&', f"{rnn_results['rmse']:.2f}", '&', f"{lstm_results['rmse']:.2f}")

# Plot yield curves for a few test examples
plot_yield_curves(
    cnn_results['denorm_predictions'],
    cnn_results['denorm_targets'],
    maturities,
    indices=[0, 1, 2],
    title='CNN-Transformer Model: Predicted vs Actual Yield Curves'
)

plot_yield_curves(
    transformer_results['denorm_predictions'],
    transformer_results['denorm_targets'],
    maturities,
    indices=[0, 1, 2],
    title='Transformer-Only Model: Predicted vs Actual Yield Curves'
)

# Plot yield curves for a few test examples
plot_yield_curves(
    rnn_results['denorm_predictions'],
    rnn_results['denorm_targets'],
    maturities,
    indices=[0, 1, 2],
    title='CNN-RNN Model: Predicted vs Actual Yield Curves'
)


# Plot yield curves for a few test examples
plot_yield_curves(
    lstm_results['denorm_predictions'],
    lstm_results['denorm_targets'],
    maturities,
    indices=[0, 1, 2],
    title='CNN-LSTM Model: Predicted vs Actual Yield Curves'
)


In [ ]:
# Analyze factors and lambda over the full training and evaluation period
# Create DataLoaders with sequential data (no shuffling) for the full dataset
full_dataset = torch.utils.data.ConcatDataset([train_dataset_with_macro, val_dataset_with_macro, test_dataset_with_macro])
full_loader = DataLoader(full_dataset, batch_size=16, shuffle=False)

# Extract factors and lambdas for each model
cnn_factors, cnn_lambdas = extract_factors_and_lambdas(cnn_transformer_model, full_loader)
transformer_factors, transformer_lambdas = extract_factors_and_lambdas(transformer_only_model, full_loader)
rnn_factors, rnn_lambdas = extract_factors_and_lambdas(cnn_rnn_model, full_loader)
lstm_factors, lstm_lambdas = extract_factors_and_lambdas(cnn_lstm_model, full_loader)


# Get all dates for the full period (accounting for lookback period)
full_dates = df.index[seq_length:len(full_dataset)+seq_length]  # Adjusted for lookback period

# Make sure the number of dates matches the number of factors
if len(full_dates) > len(cnn_factors):
    full_dates = full_dates[:len(cnn_factors)]
elif len(full_dates) < len(cnn_factors):
    cnn_factors = cnn_factors[:len(full_dates)]
    cnn_lambdas = cnn_lambdas[:len(full_dates)]
    transformer_factors = transformer_factors[:len(full_dates)]
    transformer_lambdas = transformer_lambdas[:len(full_dates)]
    rnn_factors = rnn_factors[:len(full_dates)]
    rnn_lambdas = rnn_lambdas[:len(full_dates)]
    lstm_factors = lstm_factors[:len(full_dates)]
    lstm_lambdas = lstm_lambdas[:len(full_dates)]

# Create a figure with good size for 4 subplots
plt.figure(figsize=(12, 15))

# Plot comparison for each factor
factor_names = ['Level', 'Slope', 'Curvature', 'Lambda']
factor_data = [
    [cnn_factors[:, i] for i in range(3)] + [cnn_lambdas],
    [transformer_factors[:, i] for i in range(3)] + [transformer_lambdas],
    [rnn_factors[:, i] for i in range(3)] + [rnn_lambdas],
    [lstm_factors[:, i] for i in range(3)] + [lstm_lambdas]
]

model_names = ['CNN-Transformer', 'Transformer-Only', 'CNN-RNN', 'CNN-LSTM']
colors = ['b', 'r', 'g', 'purple']  # Note: 'p' is not a valid color; use 'purple' instead

# Make sure dates are datetime objects
if isinstance(full_dates[0], str):
    full_dates = pd.to_datetime(full_dates)

# Create subplots with shared x-axis
fig, axs = plt.subplots(4, 1, figsize=(15, 15), sharex=True)

# Plot each factor
for i, factor_name in enumerate(factor_names):
    ax = axs[i]
    for j, model_name in enumerate(model_names):
        # Fix the data access based on the structure
        data = factor_data[j][i]
        ax.plot(full_dates, data, color=colors[j], label=model_name, linewidth=1.5)

    ax.set_title(f'{factor_name} Factor Comparison', fontsize=14)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.legend(fontsize=10)

    # Add horizontal line at zero for reference
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)

    # Set y-axis limits to be consistent across models
    if i < 3:  # For Level, Slope, Curvature
        all_values = np.concatenate([factor_data[j][i] for j in range(len(model_names))])
        min_val = np.min(all_values)
        max_val = np.max(all_values)
        # Add 10% margin
        margin = (max_val - min_val) * 0.1
        ax.set_ylim(min_val - margin, max_val + margin)

# Format x-axis to show years only
years = mdates.YearLocator(4)  # Show every 4 years
years_fmt = mdates.DateFormatter("'%y")  # Format as 'YY (last two digits with apostrophe)

# Apply formatting to x-axis
for ax in axs:
    ax.xaxis.set_major_locator(years)
    ax.xaxis.set_major_formatter(years_fmt)
    ax.set_xlim(full_dates[0], full_dates[-1])  # Ensure x-axis limits match the data

# Add a title and adjust layout
plt.suptitle('Yield Curve Factors Comparison Between Models', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.savefig('factor_comparison_between_models_macro_6.png', dpi=300, bbox_inches='tight')
plt.show();

In [ ]:
# Step 9: Multi-step forecasting
# Get first batch from test loader for demonstration
test_batch = next(iter(test_loader))
X_test, y_test = test_batch

# Generate multi-step forecasts
cnn_multi_forecasts = multi_step_forecast(cnn_transformer_model, X_test, pred_horizon=4, use_macro=True, num_macro=macro_vars)
transformer_multi_forecasts = multi_step_forecast(transformer_only_model, X_test, pred_horizon=4, use_macro=True, num_macro=macro_vars)
rnn_multi_forecasts = multi_step_forecast(cnn_rnn_model, X_test, pred_horizon=4, use_macro=True, num_macro=macro_vars)
lstm_multi_forecasts = multi_step_forecast(cnn_lstm_model, X_test, pred_horizon=4, use_macro=True, num_macro=macro_vars)


# Plot multi-step forecasts for the first sample - CNN-Transformer model
plt.figure(figsize=(15, 10))

# First check how many target horizons are available
available_horizons = y_test.shape[1]
print(f"Available target horizons in y_test: {available_horizons}")

for h in range(min(4, available_horizons-1)):  # Limit to available horizons
    plt.subplot(3, 2, h+1)

    # Predicted values
    plt.plot(maturities, yield_scaler.inverse_transform(cnn_multi_forecasts[h][0].reshape(-1, 1)).reshape(-1), 'r-', label=f'Predicted (t+{h+1})')

    # Check if this target horizon is available before plotting
    actual_target_idx = h+1  # This accounts for the change in YCdataset
    if actual_target_idx < available_horizons:
        plt.plot(maturities, yield_scaler.inverse_transform(y_test[0, actual_target_idx].numpy().reshape(-1, 1)).reshape(-1), 'b-', label=f'Actual (t+{h+1})')
    else:
        print(f"Warning: Target horizon t+{h+1} not available in y_test")

    plt.title(f'{h+1}-Month Ahead Forecast')
    tick_positions = range(0, len(maturities), 2)  # Every other index
    plt.title(f'{h+1}-Month Ahead Forecast')
    plt.xlabel('Maturity')
    plt.ylabel('Yield (%)')

    # Set custom x-axis ticks
    plt.xticks([0, 5, 10, 15, 20, 25, 30])
    plt.legend()

plt.tight_layout()
plt.savefig('cnn_transformer_multi_step_forecasts_macro_6.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot multi-step forecasts for the first sample - Transformer-Only model
plt.figure(figsize=(15, 10))

for h in range(min(4, available_horizons-1)):  # Limit to available horizons
    plt.subplot(3, 2, h+1)

    # Predicted values
    plt.plot(maturities, yield_scaler.inverse_transform(transformer_multi_forecasts[h][0].reshape(-1, 1)).reshape(-1), 'r-', label=f'Predicted (t+{h+1})')

    # Check if this target horizon is available before plotting
    actual_target_idx = h+1  # This accounts for the change in YCdataset
    if actual_target_idx < available_horizons:
        plt.plot(maturities, yield_scaler.inverse_transform(y_test[0, actual_target_idx].numpy().reshape(-1, 1)).reshape(-1), 'b-', label=f'Actual (t+{h+1})')
    else:
        print(f"Warning: Target horizon t+{h+1} not available in y_test")

    plt.title(f'{h+1}-Month Ahead Forecast')
    tick_positions = range(0, len(maturities), 2)  # Every other index
    plt.title(f'{h+1}-Month Ahead Forecast')
    plt.xlabel('Maturity')
    plt.ylabel('Yield (%)')

    # Set custom x-axis ticks
    plt.xticks([0, 5, 10, 15, 20, 25, 30])
    plt.legend()

plt.tight_layout()
plt.savefig('transformer_only_multi_step_forecasts_macro_6.png', dpi=300, bbox_inches='tight')
plt.show();


# Plot multi-step forecasts for the first sample - CNN-RNN model
plt.figure(figsize=(15, 10))

for h in range(min(4, available_horizons-1)):  # Limit to available horizons
    plt.subplot(3, 2, h+1)

    # Predicted values
    plt.plot(maturities, yield_scaler.inverse_transform(rnn_multi_forecasts[h][0].reshape(-1, 1)).reshape(-1), 'r-', label=f'Predicted (t+{h+1})')

    # Check if this target horizon is available before plotting
    actual_target_idx = h+1  # This accounts for the change in YCdataset
    if actual_target_idx < available_horizons:
        plt.plot(maturities, yield_scaler.inverse_transform(y_test[0, actual_target_idx].numpy().reshape(-1, 1)).reshape(-1), 'b-', label=f'Actual (t+{h+1})')
    else:
        print(f"Warning: Target horizon t+{h+1} not available in y_test")

    plt.title(f'{h+1}-Month Ahead Forecast')
    tick_positions = range(0, len(maturities), 2)  # Every other index
    plt.title(f'{h+1}-Month Ahead Forecast')
    plt.xlabel('Maturity')
    plt.ylabel('Yield (%)')

    # Set custom x-axis ticks
    plt.xticks([0, 5, 10, 15, 20, 25, 30])
    plt.legend()

plt.tight_layout()
plt.savefig('cnn_rnn_multi_step_forecasts_macro_6.png', dpi=300, bbox_inches='tight')
plt.show();

# Plot multi-step forecasts for the first sample - CNN-LSTM model
plt.figure(figsize=(15, 10))

for h in range(min(4, available_horizons-1)):  # Limit to available horizons
    plt.subplot(3, 2, h+1)

    # Predicted values
    plt.plot(maturities, yield_scaler.inverse_transform(lstm_multi_forecasts[h][0].reshape(-1, 1)).reshape(-1), 'r-', label=f'Predicted (t+{h+1})')

    # Check if this target horizon is available before plotting
    actual_target_idx = h+1  # This accounts for the change in YCdataset
    if actual_target_idx < available_horizons:
        plt.plot(maturities, yield_scaler.inverse_transform(y_test[0, actual_target_idx].numpy().reshape(-1, 1)).reshape(-1), 'b-', label=f'Actual (t+{h+1})')
    else:
        print(f"Warning: Target horizon t+{h+1} not available in y_test")

    plt.title(f'{h+1}-Month Ahead Forecast')
    tick_positions = range(0, len(maturities), 2)  # Every other index
    plt.title(f'{h+1}-Month Ahead Forecast')
    plt.xlabel('Maturity')
    plt.ylabel('Yield (%)')

    # Set custom x-axis ticks
    plt.xticks([0, 5, 10, 15, 20, 25, 30])
    plt.legend()


plt.tight_layout()
plt.savefig('cnn_lstm_multi_step_forecasts_macro_6.png', dpi=300, bbox_inches='tight')
plt.show();

In [ ]:
# Step 10: Evaluate multi-step forecasting performance
# Set parameters
pred_horizon = 6
yield_cols = train_dataset_with_macro.yield_cols
yield_dim = len(yield_cols)  # Number of yield curve maturities
num_macro = macro_vars  # Number of macro variables

# Full evaluation using the evaluate_multi_step_forecast function
print("\nCNN-Transformer Multi-step Evaluation:")
cnn_multi_results = evaluate_multi_step_forecast(
    model=cnn_transformer_model,
    test_loader=test_loader,
    yield_scaler=yield_scaler,
    pred_horizon=pred_horizon,
    yield_dim=yield_dim,
    use_macro=True,
    num_macro=num_macro,
    denormalize=True
)

print("\nTransformer-Only Multi-step Evaluation:")
transformer_multi_results = evaluate_multi_step_forecast(
    model=transformer_only_model,
    test_loader=test_loader,
    yield_scaler=yield_scaler,
    pred_horizon=pred_horizon,
    yield_dim=yield_dim,
    use_macro=True,
    num_macro=num_macro,
    denormalize=True
)

print("\nCNN-RNN Multi-step Evaluation:")
rnn_multi_results = evaluate_multi_step_forecast(
    model=cnn_rnn_model,
    test_loader=test_loader,
    yield_scaler=yield_scaler,
    pred_horizon=pred_horizon,
    yield_dim=yield_dim,
    use_macro=True,
    num_macro=num_macro,
    denormalize=True
)


print("\nCNN-LSTM Multi-step Evaluation:")
lstm_multi_results = evaluate_multi_step_forecast(
    model=cnn_lstm_model,
    test_loader=test_loader,
    yield_scaler=yield_scaler,
    pred_horizon=pred_horizon,
    yield_dim=yield_dim,
    use_macro=True,
    num_macro=num_macro,
    denormalize=True
)


# Print RMSE by forecast horizon for each model
print("\nRMSE by Forecast Horizon (in basis points):")
print(f"{'Horizon':<10} {'CNN-Transformer':<20} {'Transformer-Only':<20} {'CNN-RNN':<20} {'CNN-LSTM':<20}")
print("-" * 50)
for h in range(pred_horizon):
    cnn_rmse = cnn_multi_results['rmse_by_horizon'][h] * 100  # Convert to basis points
    transformer_rmse = transformer_multi_results['rmse_by_horizon'][h] * 100  # Convert to basis points
    rnn_rmse = rnn_multi_results['rmse_by_horizon'][h] * 100  # Convert to basis points
    lstm_rmse = lstm_multi_results['rmse_by_horizon'][h] * 100  # Convert to basis points
    print(f"{h+1:<10} {cnn_rmse:<20.2f} {transformer_rmse:<20.2f} {rnn_rmse:<20.2f} {lstm_rmse:<20.2f}")

# Print overall RMSE for each model
print("\nOverall Multi-step RMSE (in basis points):")
print(f"CNN-Transformer: {cnn_multi_results['rmse'] * 100:.2f}")
print(f"Transformer-Only: {transformer_multi_results['rmse'] * 100:.2f}")
print(f"CNN-RNN: {rnn_multi_results['rmse'] * 100:.2f}")
print(f"CNN-LSTM: {lstm_multi_results['rmse'] * 100:.2f}")


# Visualize multi-step forecasting performance
plt.figure(figsize=(15, 10))

# Plot RMSE by horizon
plt.subplot(2, 2, 1)
horizons = range(1, pred_horizon + 1)
plt.plot(horizons, [rmse * 100 for rmse in cnn_multi_results['rmse_by_horizon']], 'o-', label='CNN-Transformer')
plt.plot(horizons, [rmse * 100 for rmse in transformer_multi_results['rmse_by_horizon']], 's-', label='Transformer-Only')
plt.plot(horizons, [rmse * 100 for rmse in rnn_multi_results['rmse_by_horizon']], 's-', label='CNN-RNN')
plt.plot(horizons, [rmse * 100 for rmse in lstm_multi_results['rmse_by_horizon']], 's-', label='CNN-LSTM')
plt.xlabel('Forecast Horizon (months)')
plt.ylabel('RMSE (basis points)')
plt.title('RMSE by Forecast Horizon (lookback window=9, with Macro)')
plt.legend()

# Plot RMSE by maturity for each model
plt.subplot(2, 2, 2)
maturities = yield_cols
plt.plot(range(len(maturities)), cnn_multi_results['rmse_by_maturity'] * 100, 'o-', label='CNN-Transformer')
plt.plot(range(len(maturities)), transformer_multi_results['rmse_by_maturity'] * 100, 's-', label='Transformer-Only')
plt.plot(range(len(maturities)), rnn_multi_results['rmse_by_maturity'] * 100, '^-', label='CNN-RNN')
plt.plot(range(len(maturities)), lstm_multi_results['rmse_by_maturity'] * 100, 'd-', label='CNN-LSTM')
plt.xlabel('Maturity')
plt.ylabel('RMSE (basis points)')
plt.title('RMSE by Maturity (lookback window=9, with Macro)')

tick_positions = range(0, len(maturities), 5)  # Every other index
tick_labels = [maturities[i].replace('Y', '') for i in tick_positions]  # Get corresponding maturity values
plt.xticks(tick_positions, tick_labels)
plt.xlabel('Maturity (y)')
plt.savefig('multi_step_forecasts_macro_6.png', dpi=300, bbox_inches='tight')
plt.legend()

plt.show();

## Without Macro version

In [ ]:
# Function to fix seed
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Create data loaders
batch_size = 16
train_loader_wo = DataLoader(train_dataset_without_macro, batch_size=batch_size, shuffle=True, worker_init_fn=seed_worker,
                          generator=torch.Generator().manual_seed(SEED))
val_loader_wo = DataLoader(val_dataset_without_macro, batch_size=batch_size, worker_init_fn=seed_worker,
                          generator=torch.Generator().manual_seed(SEED))
test_loader_wo = DataLoader(test_dataset_without_macro, batch_size=batch_size, worker_init_fn=seed_worker,
                          generator=torch.Generator().manual_seed(SEED))

# Define model parameters
input_dim = train_dataset_without_macro[0][0].shape[1]  # Total number of features (yields + macro)
yield_dim = len(train_dataset_with_macro.yield_cols)  # Number of yield maturities
macro_dim = len(train_dataset_with_macro.macro_cols)  # Number of macro features
seq_length = train_dataset_with_macro.seq_length
maturities = [float(col.replace('Y', '')) for col in train_dataset_with_macro.yield_cols]  # Extract maturities from column names

# Create models (time-invariant lambda)
cnn_transformer_model_wo = DLNS_CNNTransformer(
    input_dim=input_dim,
    yield_dim=yield_dim,
    macro_dim=macro_dim,
    maturities=maturities,
    hidden_dim=64,
    time_varying_decay=True,
    cnn_out_channels=32,
    cnn_kernel_size=3,
    transformer_dim=64,
    nhead=4,
    num_transformer_layers=2,
    dropout=0.1,
    seq_length=seq_length,
    use_macro=False
)

transformer_only_model_wo = DLNSTransformerOnly(
    input_dim=input_dim,
    maturities=maturities,
    hidden_dim=64,
    time_varying_decay=True,
    transformer_dim=64,
    nhead=4,
    num_transformer_layers=2,
    dropout=0.1,
    seq_length=seq_length

)

cnn_rnn_model_wo = DLNS_CNNRNN(
    input_dim=input_dim,
    yield_dim=yield_dim,
    macro_dim=macro_dim,
    maturities=maturities,
    hidden_dim=64,
    time_varying_decay=True,
    cnn_out_channels=32,
    cnn_kernel_size=3,
    rnn_hidden_dim=64,
    num_rnn_layers=2,
    dropout=0.1,
    seq_length=seq_length,
    use_macro=False,
    bidirectional=False
)


cnn_lstm_model_wo = DLNS_CNNLSTM(
    input_dim=input_dim,
    yield_dim=yield_dim,
    macro_dim=macro_dim,
    maturities=maturities,
    hidden_dim=64,
    time_varying_decay=True,
    cnn_out_channels=32,
    cnn_kernel_size=3,
    lstm_hidden_dim=64,
    num_lstm_layers=2,
    dropout=0.1,
    seq_length=seq_length,
    use_macro=False
)


In [ ]:
## Train models
# CNN-Transformer model
cnn_transformer_model_wo, cnn_history_wo = train_model(
    cnn_transformer_model_wo,
    train_loader_wo,
    val_loader_wo,
    criterion=torch.nn.MSELoss(),
    learning_rate=0.001,
    weight_decay=1e-5,
    n_epochs=100,
    patience=30,
    model_path='cnn_transformer_model_wo.pth'
)


In [ ]:
# Plot training history
plot_training_history(cnn_history_wo)


In [ ]:
# Transformer-only model
transformer_only_model_wo, transformer_history_wo = train_model(
    transformer_only_model_wo,
    train_loader_wo,
    val_loader_wo,
    criterion=torch.nn.MSELoss(),
    learning_rate=0.001,
    weight_decay=1e-5,
    n_epochs=100,
    patience=30,
    model_path='transformer_only_model_wo.pth'
)

In [ ]:
# Plot training history
plot_training_history(transformer_history_wo)


In [ ]:
## Train models
# CNN-RNN Model
cnn_rnn_model_wo, rnn_history_wo = train_model(
    cnn_rnn_model_wo,
    train_loader_wo,
    val_loader_wo,
    criterion=torch.nn.MSELoss(),
    learning_rate=0.001,
    weight_decay=1e-5,
    n_epochs=100,
    patience=30,
    model_path='cnn_rnn_model_wo.pth'
)


In [ ]:
# Plot training history
plot_training_history(rnn_history_wo)


In [ ]:
## Train models
# CNN-LSTM Model
cnn_lstm_model_wo, lstm_history_wo = train_model(
    cnn_lstm_model_wo,
    train_loader_wo,
    val_loader_wo,
    criterion=torch.nn.MSELoss(),
    learning_rate=0.001,
    weight_decay=1e-5,
    n_epochs=100,
    patience=30,
    model_path='cnn_lstm_model_wo.pth'
)


In [ ]:
# Plot training history
plot_training_history(lstm_history_wo)


In [ ]:
# Step 5: Evaluate models
# CNN-Transformer model
cnn_results_wo = denormalize_and_evaluate(
    cnn_transformer_model_wo,
    test_loader_wo,
    yield_scaler,
    maturities
)

# Transformer-only model
transformer_results_wo = denormalize_and_evaluate(
    transformer_only_model_wo,
    test_loader_wo,
    yield_scaler,
    maturities
)


# CNN-RNN model
rnn_results_wo = denormalize_and_evaluate(
    cnn_rnn_model_wo,
    test_loader_wo,
    yield_scaler,
    maturities
)

# CNN-LSTM model
lstm_results_wo = denormalize_and_evaluate(
    cnn_lstm_model_wo,
    test_loader_wo,
    yield_scaler,
    maturities
)


In [ ]:
# Step 6: Compare model performance
print(f"CNN-Transformer Model RMSE: {cnn_results_wo['rmse']:.6f}")
print(f"Transformer-Only Model RMSE: {transformer_results_wo['rmse']:.6f}")
print(f"CNN-RNN Model RMSE: {rnn_results_wo['rmse']:.6f}")
print(f"CNN-LSTM Model RMSE: {lstm_results_wo['rmse']:.6f}")


# Plot yield curves for a few test examples
plot_yield_curves(
    cnn_results_wo['denorm_predictions'],
    cnn_results_wo['denorm_targets'],
    maturities,
    indices=[0, 1, 2],
    title='CNN-Transformer Model: Predicted vs Actual Yield Curves'
)

plot_yield_curves(
    transformer_results_wo['denorm_predictions'],
    transformer_results_wo['denorm_targets'],
    maturities,
    indices=[0, 1, 2],
    title='Transformer-Only Model: Predicted vs Actual Yield Curves'
)

# Plot yield curves for a few test examples
plot_yield_curves(
    rnn_results_wo['denorm_predictions'],
    rnn_results_wo['denorm_targets'],
    maturities,
    indices=[0, 1, 2],
    title='CNN-RNN Model: Predicted vs Actual Yield Curves'
)


# Plot yield curves for a few test examples
plot_yield_curves(
    lstm_results_wo['denorm_predictions'],
    lstm_results_wo['denorm_targets'],
    maturities,
    indices=[0, 1, 2],
    title='CNN-LSTM Model: Predicted vs Actual Yield Curves'
)


In [ ]:
# Analyze factors and lambda over the full training and evaluation period
# Create DataLoaders with sequential data (no shuffling) for the full dataset
full_dataset = torch.utils.data.ConcatDataset([train_dataset_without_macro, val_dataset_without_macro, test_dataset_without_macro])
full_loader = DataLoader(full_dataset, batch_size=16, shuffle=False)

# Extract factors and lambdas for each model
cnn_factors, cnn_lambdas = extract_factors_and_lambdas(cnn_transformer_model_wo, full_loader)
transformer_factors, transformer_lambdas = extract_factors_and_lambdas(transformer_only_model_wo, full_loader)
rnn_factors, rnn_lambdas = extract_factors_and_lambdas(cnn_rnn_model_wo, full_loader)
lstm_factors, lstm_lambdas = extract_factors_and_lambdas(cnn_lstm_model_wo, full_loader)


# Get all dates for the full period (accounting for lookback period)
full_dates = df.index[seq_length:len(full_dataset)+seq_length]  # Adjusted for lookback period

# Make sure the number of dates matches the number of factors
if len(full_dates) > len(cnn_factors):
    full_dates = full_dates[:len(cnn_factors)]
elif len(full_dates) < len(cnn_factors):
    cnn_factors = cnn_factors[:len(full_dates)]
    cnn_lambdas = cnn_lambdas[:len(full_dates)]
    transformer_factors = transformer_factors[:len(full_dates)]
    transformer_lambdas = transformer_lambdas[:len(full_dates)]
    rnn_factors = rnn_factors[:len(full_dates)]
    rnn_lambdas = rnn_lambdas[:len(full_dates)]
    lstm_factors = lstm_factors[:len(full_dates)]
    lstm_lambdas = lstm_lambdas[:len(full_dates)]

# Create a figure with good size for 4 subplots
plt.figure(figsize=(12, 15))

# Plot comparison for each factor
factor_names = ['Level', 'Slope', 'Curvature', 'Lambda']
factor_data = [
    [cnn_factors[:, i] for i in range(3)] + [cnn_lambdas],
    [transformer_factors[:, i] for i in range(3)] + [transformer_lambdas],
    [rnn_factors[:, i] for i in range(3)] + [rnn_lambdas],
    [lstm_factors[:, i] for i in range(3)] + [lstm_lambdas]
]

model_names = ['CNN-Transformer', 'Transformer-Only', 'CNN-RNN', 'CNN-LSTM']
colors = ['b', 'r', 'g', 'purple']  # Note: 'p' is not a valid color; use 'purple' instead

# Make sure dates are datetime objects
if isinstance(full_dates[0], str):
    full_dates = pd.to_datetime(full_dates)

# Create subplots with shared x-axis
fig, axs = plt.subplots(4, 1, figsize=(15, 15), sharex=True)

# Plot each factor
for i, factor_name in enumerate(factor_names):
    ax = axs[i]
    for j, model_name in enumerate(model_names):
        # Fix the data access based on the structure
        data = factor_data[j][i]
        ax.plot(full_dates, data, color=colors[j], label=model_name, linewidth=1.5)

    ax.set_title(f'{factor_name} Factor Comparison', fontsize=14)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.legend(fontsize=10)

    # Add horizontal line at zero for reference
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)

    # Set y-axis limits to be consistent across models
    if i < 3:  # For Level, Slope, Curvature
        all_values = np.concatenate([factor_data[j][i] for j in range(len(model_names))])
        min_val = np.min(all_values)
        max_val = np.max(all_values)
        # Add 10% margin
        margin = (max_val - min_val) * 0.1
        ax.set_ylim(min_val - margin, max_val + margin)

# Format x-axis to show years only
years = mdates.YearLocator(4)  # Show every 4 years
years_fmt = mdates.DateFormatter("'%y")  # Format as 'YY (last two digits with apostrophe)

# Apply formatting to x-axis
for ax in axs:
    ax.xaxis.set_major_locator(years)
    ax.xaxis.set_major_formatter(years_fmt)
    ax.set_xlim(full_dates[0], full_dates[-1])  # Ensure x-axis limits match the data

# Add a title and adjust layout
plt.suptitle('Yield Curve Factors Comparison Between Models', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.savefig('factor_comparison_between_models_no_6.png', dpi=300, bbox_inches='tight')
plt.show();

In [ ]:
# Step 9: Multi-step forecasting
# Get first batch from test loader for demonstration
test_batch = next(iter(test_loader_wo))
X_test, y_test = test_batch

# Generate multi-step forecasts
cnn_multi_forecasts = multi_step_forecast(cnn_transformer_model_wo, X_test, pred_horizon=4, use_macro=False, num_macro=macro_vars)
transformer_multi_forecasts = multi_step_forecast(transformer_only_model_wo, X_test, pred_horizon=4, use_macro=False, num_macro=macro_vars)
rnn_multi_forecasts = multi_step_forecast(cnn_rnn_model_wo, X_test, pred_horizon=4, use_macro=False, num_macro=macro_vars)
lstm_multi_forecasts = multi_step_forecast(cnn_lstm_model_wo, X_test, pred_horizon=4, use_macro=False, num_macro=macro_vars)


# Plot multi-step forecasts for the first sample - CNN-Transformer model
plt.figure(figsize=(15, 10))

# First check how many target horizons are available
available_horizons = y_test.shape[1]
print(f"Available target horizons in y_test: {available_horizons}")

for h in range(min(4, available_horizons-1)):  # Limit to available horizons
    plt.subplot(3, 2, h+1)

    # Predicted values
    plt.plot(maturities, yield_scaler.inverse_transform(cnn_multi_forecasts[h][0].reshape(-1, 1)).reshape(-1), 'r-', label=f'Predicted (t+{h+1})')

    # Check if this target horizon is available before plotting
    actual_target_idx = h+1  # This accounts for the change in YCdataset
    if actual_target_idx < available_horizons:
        plt.plot(maturities, yield_scaler.inverse_transform(y_test[0, actual_target_idx].numpy().reshape(-1, 1)).reshape(-1), 'b-', label=f'Actual (t+{h+1})')
    else:
        print(f"Warning: Target horizon t+{h+1} not available in y_test")

    plt.title(f'{h+1}-Month Ahead Forecast')
    plt.xlabel('Maturity')
    plt.ylabel('Yield (%)')

    # Set custom x-axis ticks
    plt.xticks([0, 5, 10, 15, 20, 25, 30])
    plt.legend()


plt.tight_layout()
plt.savefig('cnn_transformer_multi_step_forecasts_no_6.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot multi-step forecasts for the first sample - Transformer-Only model
plt.figure(figsize=(15, 10))

for h in range(min(4, available_horizons-1)):  # Limit to available horizons
    plt.subplot(3, 2, h+1)

    # Predicted values
    plt.plot(maturities, yield_scaler.inverse_transform(transformer_multi_forecasts[h][0].reshape(-1, 1)).reshape(-1), 'r-', label=f'Predicted (t+{h+1})')

    # Check if this target horizon is available before plotting
    actual_target_idx = h+1  # This accounts for the change in YCdataset
    if actual_target_idx < available_horizons:
        plt.plot(maturities, yield_scaler.inverse_transform(y_test[0, actual_target_idx].numpy().reshape(-1, 1)).reshape(-1), 'b-', label=f'Actual (t+{h+1})')
    else:
        print(f"Warning: Target horizon t+{h+1} not available in y_test")

    plt.title(f'{h+1}-Month Ahead Forecast')
    plt.xlabel('Maturity')
    plt.ylabel('Yield (%)')

    # Set custom x-axis ticks
    plt.xticks([0, 5, 10, 15, 20, 25, 30])
    plt.legend()


plt.tight_layout()
plt.savefig('transformer_only_multi_step_forecasts_no_6.png', dpi=300, bbox_inches='tight')
plt.show();


# Plot multi-step forecasts for the first sample - CNN-RNN model
plt.figure(figsize=(15, 10))

for h in range(min(4, available_horizons-1)):  # Limit to available horizons
    plt.subplot(3, 2, h+1)

    # Predicted values
    plt.plot(maturities, yield_scaler.inverse_transform(rnn_multi_forecasts[h][0].reshape(-1, 1)).reshape(-1), 'r-', label=f'Predicted (t+{h+1})')

    # Check if this target horizon is available before plotting
    actual_target_idx = h+1  # This accounts for the change in YCdataset
    if actual_target_idx < available_horizons:
        plt.plot(maturities, yield_scaler.inverse_transform(y_test[0, actual_target_idx].numpy().reshape(-1, 1)).reshape(-1), 'b-', label=f'Actual (t+{h+1})')
    else:
        print(f"Warning: Target horizon t+{h+1} not available in y_test")

    plt.title(f'{h+1}-Month Ahead Forecast')
    plt.xlabel('Maturity')
    plt.ylabel('Yield (%)')

    # Set custom x-axis ticks
    plt.xticks([0, 5, 10, 15, 20, 25, 30])

    plt.legend()


plt.tight_layout()
plt.savefig('cnn_rnn_multi_step_forecasts_no_6.png', dpi=300, bbox_inches='tight')
plt.show();

# Plot multi-step forecasts for the first sample - CNN-LSTM model
plt.figure(figsize=(15, 10))

for h in range(min(4, available_horizons-1)):  # Limit to available horizons
    plt.subplot(3, 2, h+1)

    # Predicted values
    plt.plot(maturities, yield_scaler.inverse_transform(lstm_multi_forecasts[h][0].reshape(-1, 1)).reshape(-1), 'r-', label=f'Predicted (t+{h+1})')

    # Check if this target horizon is available before plotting
    actual_target_idx = h+1  # This accounts for the change in YCdataset
    if actual_target_idx < available_horizons:
        plt.plot(maturities, yield_scaler.inverse_transform(y_test[0, actual_target_idx].numpy().reshape(-1, 1)).reshape(-1), 'b-', label=f'Actual (t+{h+1})')
    else:
        print(f"Warning: Target horizon t+{h+1} not available in y_test")

    plt.title(f'{h+1}-Month Ahead Forecast')
    plt.xlabel('Maturity')
    plt.ylabel('Yield (%)')

    # Set custom x-axis ticks
    plt.xticks([0, 5, 10, 15, 20, 25, 30])

    plt.legend()



plt.tight_layout()
plt.savefig('cnn_lstm_multi_step_forecasts_no_6.png', dpi=300, bbox_inches='tight')
plt.show();

In [ ]:
# Step 10: Evaluate multi-step forecasting performance
# Set parameters
pred_horizon = 6
yield_cols = train_dataset_with_macro.yield_cols
yield_dim = len(yield_cols)  # Number of yield curve maturities
num_macro = macro_vars  # Number of macro variables

# Full evaluation using the evaluate_multi_step_forecast function
print("\nCNN-Transformer Multi-step Evaluation:")
cnn_multi_results = evaluate_multi_step_forecast(
    model=cnn_transformer_model_wo,
    test_loader=test_loader_wo,
    yield_scaler=yield_scaler,
    pred_horizon=pred_horizon,
    yield_dim=yield_dim,
    use_macro=False,
    num_macro=num_macro,
    denormalize=True
)

print("\nTransformer-Only Multi-step Evaluation:")
transformer_multi_results = evaluate_multi_step_forecast(
    model=transformer_only_model_wo,
    test_loader=test_loader_wo,
    yield_scaler=yield_scaler,
    pred_horizon=pred_horizon,
    yield_dim=yield_dim,
    use_macro=False,
    num_macro=num_macro,
    denormalize=True
)

print("\nCNN-RNN Multi-step Evaluation:")
rnn_multi_results = evaluate_multi_step_forecast(
    model=cnn_rnn_model_wo,
    test_loader=test_loader_wo,
    yield_scaler=yield_scaler,
    pred_horizon=pred_horizon,
    yield_dim=yield_dim,
    use_macro=False,
    num_macro=num_macro,
    denormalize=True
)


print("\nCNN-LSTM Multi-step Evaluation:")
lstm_multi_results = evaluate_multi_step_forecast(
    model=cnn_lstm_model_wo,
    test_loader=test_loader_wo,
    yield_scaler=yield_scaler,
    pred_horizon=pred_horizon,
    yield_dim=yield_dim,
    use_macro=False,
    num_macro=num_macro,
    denormalize=True
)


# Print RMSE by forecast horizon for each model
print("\nRMSE by Forecast Horizon (in basis points):")
print(f"{'Horizon':<10} {'CNN-Transformer':<20} {'Transformer-Only':<20} {'CNN-RNN':<20} {'CNN-LSTM':<20}")
print("-" * 50)
for h in range(pred_horizon):
    cnn_rmse = cnn_multi_results['rmse_by_horizon'][h] * 100  # Convert to basis points
    transformer_rmse = transformer_multi_results['rmse_by_horizon'][h] * 100  # Convert to basis points
    rnn_rmse = rnn_multi_results['rmse_by_horizon'][h] * 100  # Convert to basis points
    lstm_rmse = lstm_multi_results['rmse_by_horizon'][h] * 100  # Convert to basis points
    print(f"{h+1:<10} {cnn_rmse:<20.2f} {transformer_rmse:<20.2f} {rnn_rmse:<20.2f} {lstm_rmse:<20.2f}")

# Print overall RMSE for each model
print("\nOverall Multi-step RMSE (in basis points):")
print(f"CNN-Transformer: {cnn_multi_results['rmse'] * 100:.2f}")
print(f"Transformer-Only: {transformer_multi_results['rmse'] * 100:.2f}")
print(f"CNN-RNN: {rnn_multi_results['rmse'] * 100:.2f}")
print(f"CNN-LSTM: {lstm_multi_results['rmse'] * 100:.2f}")


# Visualize multi-step forecasting performance
plt.figure(figsize=(15, 10))

# Plot RMSE by horizon
plt.subplot(2, 2, 1)
horizons = range(1, pred_horizon + 1)
plt.plot(horizons, [rmse * 100 for rmse in cnn_multi_results['rmse_by_horizon']], 'o-', label='CNN-Transformer')
plt.plot(horizons, [rmse * 100 for rmse in transformer_multi_results['rmse_by_horizon']], 's-', label='Transformer-Only')
plt.plot(horizons, [rmse * 100 for rmse in rnn_multi_results['rmse_by_horizon']], 's-', label='CNN-RNN')
plt.plot(horizons, [rmse * 100 for rmse in lstm_multi_results['rmse_by_horizon']], 's-', label='CNN-LSTM')
plt.xlabel('Forecast Horizon (months)')
plt.ylabel('RMSE (basis points)')
plt.title('RMSE by Forecast Horizon (lookback window=6, without Macro)')
plt.legend()

# Plot RMSE by maturity for each model
plt.subplot(2, 2, 2)
maturities = yield_cols
plt.plot(range(len(maturities)), cnn_multi_results['rmse_by_maturity'] * 100, 'o-', label='CNN-Transformer')
plt.plot(range(len(maturities)), transformer_multi_results['rmse_by_maturity'] * 100, 's-', label='Transformer-Only')
plt.plot(range(len(maturities)), rnn_multi_results['rmse_by_maturity'] * 100, '^-', label='CNN-RNN')
plt.plot(range(len(maturities)), lstm_multi_results['rmse_by_maturity'] * 100, 'd-', label='CNN-LSTM')
tick_positions = range(0, len(maturities), 5)  # Every other index
tick_labels = [maturities[i].replace('Y', '') for i in tick_positions]  # Get corresponding maturity values
plt.xticks(tick_positions, tick_labels)

plt.xlabel('Maturity (y)')
plt.ylabel('RMSE (basis points)')
plt.title('RMSE by Maturity (lookback window=6, without Macro)')
plt.legend()
plt.savefig('multi_step_forecasts_no_6.png', dpi=300, bbox_inches='tight')
plt.show();

#### Code for plots not used in the report.

In [ ]:
# Step 7: Analyze factors and lambda
# Get test dates for plotting
test_dates = df.iloc[train_size+val_size:].index[:len(cnn_results['factors'])]

# Plot CNN-Transformer factors
plot_factors_over_time(cnn_results['factors'], cnn_results['lambdas'], test_dates)

# Plot Transformer-Only factors
plot_factors_over_time(transformer_results['factors'], transformer_results['lambdas'], test_dates)

# Plot CNN-Transformer factors
plot_factors_over_time(cnn_results['factors'], cnn_results['lambdas'], test_dates)


In [ ]:
# Step 8: Visualization
# Plot factor loadings
#plot_factor_loadings(maturities, np.mean(cnn_results['lambdas']))

# Plot 3D yield curve evolution
plot_yield_curve_3d(cnn_results['denorm_predictions'], maturities, test_dates)

# Plot error heatmap
heatmap_predictions_error(
    cnn_results['denorm_predictions'],
    cnn_results['denorm_targets'],
    maturities,
    test_dates
)